## Verify that nondimensionalized inflation/design optimization is scale-invariant
Investigate uniform scaling of the spatial dimensions and simultaneous scaling of Young's modulus + inflation pressure.

In [ ]:
import sys; sys.path.append('..')
import utils

import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities, inflation, fd_validation, sheet_optimizer, opt_config
import numpy as np

In [ ]:
import py_newton_optimizer
niter = 2000
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-9

In [ ]:
global tas, sheet_opts
tas = {}
sheet_opts = {}
isFixed = None
def quantitiesAtScale(lenScale, youngScale):
    target_surf = mesh.Mesh('../../examples/lilium.msh')
    target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
    target_surf = mesh_utilities.subdivide_loop(target_surf, 1)
    
    isheet = utils.load('data/isheet.pkl.gz')
    uv = utils.load('data/uv.pkl.gz')
    isheet.thickness = 6 / isheet.youngModulus # thickness matching the stiffnesses of 1.0 in loaded isheet

    # Apply scales
    isheet.thickness = isheet.thickness * lenScale
    isheet.youngModulus = isheet.youngModulus * youngScale
    isheet.pressure = isheet.pressure * youngScale
    isheet.mesh().setVertices(isheet.mesh().vertices() * lenScale)
    isheet.setVars(isheet.getVars() * lenScale)
    target_surf.setVertices(target_surf.vertices() * lenScale)

    bv = isheet.mesh().boundaryVertices()
    bdryVars = [isheet.varIdx(0, i, c) for i in bv for c in range(3)]

    global isFixed
    fixedVars = bdryVars
    isFixed = np.zeros(isheet.numVars(), dtype=np.bool)
    isFixed[fixedVars] = True
    
    paramSampler = field_sampler.FieldSampler(np.pad(uv * lenScale, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
    liftedSheetPositions = paramSampler.sample(isheet.mesh().vertices(), target_surf.vertices())

    isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
    
    targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)
    global tas
    tas[(lenScale, youngScale)] = targetAttractedSheet # save for later access
    
    targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True
    targetAttractedSheet.fittingWeight = 1e-6
    
    # isheet.setRelaxedStiffnessEpsilon(isheet.triEnergyDensities()[0].stiffness * 1e-6)
    isheet.setRelaxedStiffnessEpsilon(1e-6)
    cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts)
    
    origDesignMesh = isheet.mesh().copy()
    sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.NONE, detActivationThreshold=0.9,
                                                 originalDesignMesh=origDesignMesh, checkpointPath=None, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0))
    sheet_opts[(lenScale, youngScale)] = sheet_opt
    
    return (utils.allEnergies     (targetAttractedSheet),
            utils.allGradientNorms(targetAttractedSheet, ~isFixed),
            utils.allEnergies(sheet_opt.rso),
            np.linalg.norm(sheet_opt.gradient(sheet_opt.rso.getVars())),
            targetAttractedSheet.nondimensionalization.length,
            targetAttractedSheet.nondimensionalization.potentialEnergyScale(),
            targetAttractedSheet.nondimensionalization.youngsModulus)

def ratios(A, B):
    def div(a, b): return a / b if b != 0.0 else 0
    return [{k: div(qB[k], qA[k]) for k in qA} if isinstance(qA, dict) else div(qB, qA)
        for qA, qB in zip(A, B)]

In [ ]:
ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(*np.random.uniform(1.0, 3.0, 2)))

In [ ]:
#ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(2.0, 1.0))

In [ ]:
#ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(4.0, 1.0))

In [ ]:
#ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(8.0, 1.0))

In [ ]:
#ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(16.0, 1.0))

In [ ]:
#ratios(quantitiesAtScale(1.0, 1.0), quantitiesAtScale(1.05, 1.0))

In [ ]:
k = list(tas.keys())
k

## Finite difference validations on one of the scaled sheets, and sheet_opts

In [ ]:
targetAttractedSheet = tas[k[1]]
so = sheet_opts[k[1]]

In [ ]:
import fd_validation

In [ ]:
fd_validation.gradConvergencePlot(targetAttractedSheet, epsilons=np.logspace(-10, -5.5, 20))

In [ ]:
fd_validation.hessConvergencePlot(targetAttractedSheet, epsilons=np.logspace(-10, -5.5, 20))

In [ ]:
fd_validation.gradConvergencePlot(so.rso, epsilons=np.logspace(-9, -6, 30))

## Verify scale invariance of the full design optimization at an edited configuration

In [ ]:
opts.gradTol = 1e-6

In [ ]:
import copy
def scaledRSO(rso, lenScale):
    origDesignMesh = rso.originalMesh().copy()
    #lenScale = 0.1
    isheet = copy.deepcopy(rso.sheet())
    # Work-around a potential inconsistency between stiffness and (E, h) settings
    effectiveOldThickness = rso.sheet().triEnergyDensities()[0].stiffness * (6 / isheet.youngModulus)
    # Apply scales
    isheet.thickness = isheet.thickness * lenScale
    scaledRestVertices = isheet.mesh().vertices() * lenScale
    isheet.mesh().setVertices(origDesignMesh.vertices() * lenScale) # make sure nondimensionalization is done on a scaling of the original mesh
    isheet.setVars(isheet.getVars() * lenScale)
    #isheet.setRestVertexPositions(scaledRestVertices[:, 0:2])

    old_tsf = rso.targetSurfaceFitter()
    scaledTargetSurf = mesh.Mesh(old_tsf.targetSurfaceV * lenScale, old_tsf.targetSurfaceF)
    targetAttractedSheet = inflation.TargetAttractedInflation(isheet, scaledTargetSurf)
    
    isheet.thickness = effectiveOldThickness * lenScale
    isheet.setRestVertexPositions(scaledRestVertices[:, 0:2])
    
    tsf = targetAttractedSheet.targetSurfaceFitter()
    tsf.holdClosestPointsFixed = True
    tsf.closestSurfPts = old_tsf.closestSurfPts * lenScale
    targetAttractedSheet.fittingWeight = rso.targetAttractedInflation().fittingWeight
    
    #print(utils.allGradientNorms(isheet, freeVariables=freeVarMask))
    #print(utils.allGradientNorms(targetAttractedSheet, freeVariables=freeVarMask))

    fixedVars = rso.fixedEquilibriumVars()

    isheet.setRelaxedStiffnessEpsilon(1e-6)
    origDesignMesh.setVertices(origDesignMesh.vertices() * lenScale)

    sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.NONE, detActivationThreshold=0.9,
                                                 originalDesignMesh=origDesignMesh, checkpointPath=None, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0))
    return sheet_opt.rso

In [ ]:
test_sheet_opt = sheet_optimizer.load('data/sheet_opt.pkl.gz')

In [ ]:
scaled_rso = scaledRSO(test_sheet_opt.rso, 1.0)
utils.allGradientNorms(scaled_rso)

In [ ]:
scaled_rso = scaledRSO(test_sheet_opt.rso, 2.0)
utils.allGradientNorms(scaled_rso)